In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# from ydata_profiling import ProfileReport #Uses pandas_profiling new version

#matplotlib.use('TkAgg')
%matplotlib inline
sns.set(style="whitegrid")


In [ ]:
df = pd.read_excel('CaseStudy_training_data.xlsx', sheet_name='Model_data')
df.head()

1. Data exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
profile = ProfileReport(df[df.columns[1:]], title="Profiling Report")
#profile
profile.to_file('profile_report.html')

3. Data cleaning

In [ ]:
df_cleaned = df.dropna()
df_cleaned = df_cleaned.drop_duplicates()
df_cleaned = df_cleaned.drop(['ID'], axis=1)
df_cleaned = df_cleaned.rename(columns={'rx ds': 'rx_ds'})
df_cleaned.head()

4. Feature Engineering

In [ ]:
# Explore the distribution of 'rx_ds' for bucketing purposes
plt.figure(figsize=(12, 6))
sns.histplot(df_cleaned['rx_ds'], bins=30, kde=True)
plt.title('Distribution of rx_ds')
plt.xlabel('rx_ds')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Perform percentile-based bucketing for 'rx_ds'
df_features = df_cleaned.copy()
df_features['rx_ds_bucket'] = pd.qcut(
    df_cleaned['rx_ds'], 
    q=4, 
    labels=['Q1', 'Q2', 'Q3', 'Q4']
    )

In [ ]:
df_features.head()

In [ ]:
# Create a new feature that is the sum of all the binary features
binary_cols = [col for col in df_features.columns if col not in ['OD', 'rx_ds', 'rx_ds_bucket']]
df_features['binary_sum'] = df_features[binary_cols].sum(axis=1)

# Create a new feature that is the ratio of 'rx_ds' to the sum of binary features
df_features['rx_ds_to_binary_sum'] = df_features['rx_ds'] / df_features['binary_sum']

In [ ]:
df_features.head()

In [ ]:
# Perform one-hot encoding for 'rx_ds_bucket'
df_one_hot = pd.get_dummies(df_features['rx_ds_bucket'], prefix='rx_ds_bucket')
df_features = pd.concat([df_features, df_one_hot], axis=1)
df_features.drop(['rx_ds_bucket'], axis=1, inplace=True)
df_features.head()

In [ ]:
# Plotting the new correlation matrix using Seaborn
correlation_matrix = df_features.corr()

plt.figure(figsize=(20, 20))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Matrix New features')
plt.show()

In [ ]:
#df_features.to_csv('df_features.csv', index=False)

5. Model building

In [ ]:
from sklearn.model_selection import train_test_split
df_model = df_features.copy()

X = df_model.drop(['OD'], axis=1)
y = df_model['OD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Baselining
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

# Majority Class (Zero-Rule Algorithm)
majority_baseline = DummyClassifier(strategy='most_frequent', random_state=42)
majority_baseline.fit(X_train, y_train)
y_pred_majority = majority_baseline.predict(X_test)
accuracy_majority = accuracy_score(y_test, y_pred_majority)

# Stratified Random Guessing
stratified_baseline = DummyClassifier(strategy='stratified', random_state=42)
stratified_baseline.fit(X_train, y_train)
y_pred_stratified = stratified_baseline.predict(X_test)
accuracy_stratified = accuracy_score(y_test, y_pred_stratified)

baseline_accuracies = pd.DataFrame({
    'Baseline Type': ['Majority Class Prediction', 'Stratified Random Guessing'],
    'Accuracy': [accuracy_majority, accuracy_stratified]
})

baseline_accuracies

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix, roc_curve

# Initializing the model, Fitting and predicting
logistic_model = LogisticRegression(random_state=42, max_iter=1000)
logistic_model.fit(X_train, y_train)
y_pred_logistic = logistic_model.predict(X_test)

In [ ]:
# # Evaluate the model
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic)
recall_logistic = recall_score(y_test, y_pred_logistic)
f1_logistic = f1_score(y_test, y_pred_logistic)
conf_matrix_logistic = confusion_matrix(y_test, y_pred_logistic)
fpr, tpr, _ = roc_curve(y_test, logistic_model.predict_proba(X_test)[:,1])
roc_auc_logistic = roc_auc_score(y_test, logistic_model.predict_proba(X_test)[:,1])

# Calculate Positive Predictive Value (PPV) and Negative Predictive Value (NPV)
TN, FP, FN, TP = conf_matrix_logistic.ravel()
ppv = TP / (TP + FP)
npv = TN / (TN + FN)
specificity_logistic = TN / (TN + FP)

# Store the results in a DataFrame for better presentation
results_logistic = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (PPV)', 'Recall (Sensitivity)', 'Specificity', 'F1 Score', 'Positive Predictive Value (PPV)', 'Negative Predictive Value (NPV)', 'ROC AUC'],
    'Logistic Regression': [accuracy_logistic, precision_logistic, recall_logistic, specificity_logistic, f1_logistic, ppv, npv, roc_auc_logistic]
})

results_logistic

In [ ]:
# Draw Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_logistic, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No OD', 'OD'], yticklabels=['No OD', 'OD'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix for Logistic Regression')
plt.show()

In [ ]:
# Plot ROC Curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=1, label=f'ROC curve (area = {roc_auc_logistic:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

6. Model improvement

In [ ]:
# Logistic Regression with class_weight parameter
logistic_weighted = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
logistic_weighted.fit(X_train, y_train)
y_pred_weighted = logistic_weighted.predict(X_test)
recall_weighted = recall_score(y_test, y_pred_weighted)
# Predicted probabilities for the positive class
y_probs = logistic_weighted.predict_proba(X_test)[:, 1]

# Calculate updated metrics
accuracy_weighted = accuracy_score(y_test, y_pred_weighted)
precision_weighted = precision_score(y_test, y_pred_weighted)
f1_score_weighted = f1_score(y_test, y_pred_weighted)
roc_auc_weighted = roc_auc_score(y_test, y_probs)

# Calculate Positive and Negative Predictive Values
conf_matrix_weighted = confusion_matrix(y_test, y_pred_weighted)
TP_weighted = conf_matrix_weighted[1, 1]
FP_weighted = conf_matrix_weighted[0, 1]
TN_weighted = conf_matrix_weighted[0, 0]
FN_weighted = conf_matrix_weighted[1, 0]
specificity_weighted = TN_weighted / (TN_weighted + FP_weighted)

ppv_weighted = TP_weighted / (TP_weighted + FP_weighted) if (TP_weighted + FP_weighted) != 0 else 0
npv_weighted = TN_weighted / (TN_weighted + FN_weighted) if (TN_weighted + FN_weighted) != 0 else 0

updated_metrics_weighted = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (PPV)', 'Recall (Sensitivity)', 'Specificity', 'F1 Score', 
               'Positive Predictive Value (PPV)', 'Negative Predictive Value (NPV)', 'ROC AUC'],
    'Weighted': [accuracy_weighted, precision_weighted, recall_weighted, specificity_weighted, 
              f1_score_weighted, ppv_weighted, npv_weighted, roc_auc_weighted],
    'Original': [accuracy_logistic, precision_logistic, recall_logistic, specificity_logistic, f1_logistic, ppv, npv, roc_auc_logistic]
})

updated_metrics_weighted

In [ ]:
from sklearn.metrics import precision_recall_curve

# Calculate precision and recall at different thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

# Plot Precision-Recall vs Threshold chart.
plt.figure(figsize=(10, 6))
plt.title("Precision and Recall Scores as a function of the decision threshold")
plt.plot(thresholds, precision[:-1], "b--", label="Precision")
plt.plot(thresholds, recall[:-1], "g-", label="Recall")
plt.ylabel("Score")
plt.xlabel("Decision Threshold")
plt.legend(loc='best')
plt.show()

In [ ]:
# Redefine the function to find the optimal threshold for classification
def find_optimal_threshold(y_true, y_probs):
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    optimal_idx = np.argmax(tpr - fpr)
    return thresholds[optimal_idx]

# Find the optimal threshold using the function
optimal_threshold = find_optimal_threshold(y_test, y_probs)
optimal_threshold

In [ ]:
# Custom threshold to make predictions
y_pred_optimal = (y_probs >= optimal_threshold).astype(int)

7. Interpretability

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': logistic_weighted.coef_[0]
}).sort_values(by='Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance)
plt.title('Feature Importance for Logistic Regression')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

In [ ]:
# SHAP (SHapley Additive exPlanations)
import shap

explainer = shap.LinearExplainer(logistic_weighted, X_train)
shap_values = explainer.shap_values(X_test)
shap.initjs()

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
# Identifying positive or negative instances
for i in range(5):
    print(f'Instance {i}:')
    print(f'Probability of OD: {y_probs[i]:.4f}')
    print(f'Predicted class: {y_pred_optimal[i]}')
    print(f'Actual class: {y_test.iloc[i]}')
    print('-' * 50)

In [ ]:
# High confidence negative prediction
shap.force_plot(explainer.expected_value, shap_values[3,:], X_test.iloc[3,:])

In [ ]:
# High confidence positive prediction
shap.force_plot(explainer.expected_value, shap_values[2,:], X_test.iloc[2,:])

In [ ]:
# High confidence postisive but non-OD
shap.force_plot(explainer.expected_value, shap_values[1,:], X_test.iloc[1,:])

In [ ]:
import lime
from lime.lime_tabular import LimeTabularExplainer

# Initialize the explainer with your existing training data
explainer = LimeTabularExplainer(X_train.to_numpy(), 
                                 training_labels=y_train, 
                                 feature_names=X_train.columns.tolist(), 
                                 class_names=['Not Addicted', 'Addicted'], 
                                 mode='classification')

In [ ]:
# High confidence negative prediction
exp = explainer.explain_instance(X_test.iloc[3].to_numpy(), logistic_weighted.predict_proba)
exp.show_in_notebook()

In [ ]:
# High confidence positive prediction
exp = explainer.explain_instance(X_test.iloc[2].to_numpy(), logistic_weighted.predict_proba)
exp.show_in_notebook()

In [ ]:
# High confidence postisive but non-OD
exp = explainer.explain_instance(X_test.iloc[1].to_numpy(), logistic_weighted.predict_proba)
exp.show_in_notebook()

8. Fairness

In [ ]:
# Demographic Parity & Disparate Impact

# Create a DataFrame to hold the true and predicted labels along with the protected attribute
fairness_df = pd.DataFrame({'True_Labels': y_test, 'Predicted_Labels': y_pred_optimal, 'Low_inc': X_test['Low_inc']})

# Calculate the selection rate for the protected and non-protected groups
selection_rate_protected = fairness_df[(fairness_df['Low_inc'] == 1)]['Predicted_Labels'].mean()
selection_rate_non_protected = fairness_df[(fairness_df['Low_inc'] == 0)]['Predicted_Labels'].mean()

# Calculate Demographic Parity
demographic_parity_difference = selection_rate_protected - selection_rate_non_protected
demographic_parity_ratio = selection_rate_protected / selection_rate_non_protected if selection_rate_non_protected != 0 else float('inf')

pd.DataFrame({
    'Metric': ['selection_rate_protected (Favorable Outcome)', 'selection_rate_non_protected (Favorable Outcome)', 'demographic_parity_difference', 'demographic_parity_ratio (Disparate Impact)'],
    'Value': [selection_rate_protected, selection_rate_non_protected, demographic_parity_difference, demographic_parity_ratio],
})

In [ ]:
# Equalized Odds

# Calculate True Positive Rates (Recall) for the protected and non-protected groups
TPR_protected = fairness_df[(fairness_df['Low_inc'] == 1) & (fairness_df['True_Labels'] == 1)]['Predicted_Labels'].mean()
TPR_non_protected = fairness_df[(fairness_df['Low_inc'] == 0) & (fairness_df['True_Labels'] == 1)]['Predicted_Labels'].mean()

# Calculate False Positive Rates for the protected and non-protected groups
FPR_protected = fairness_df[(fairness_df['Low_inc'] == 1) & (fairness_df['True_Labels'] == 0)]['Predicted_Labels'].mean()
FPR_non_protected = fairness_df[(fairness_df['Low_inc'] == 0) & (fairness_df['True_Labels'] == 0)]['Predicted_Labels'].mean()

# Calculate Equalized Odds Differences
equalized_odds_difference_TPR = TPR_protected - TPR_non_protected
equalized_odds_difference_FPR = FPR_protected - FPR_non_protected

pd.DataFrame({
    'Metric': ['TPR_protected (Recall)', 'TPR_non_protected', 'FPR_protected', 'FPR_non_protected', 'equalized_odds_difference_TPR', 'equalized_odds_difference_FPR'],
    'Value': [TPR_protected, TPR_non_protected, FPR_protected, FPR_non_protected, equalized_odds_difference_TPR, equalized_odds_difference_FPR],
})
